In [1]:
# ============================================================
# FINAL REPORT TABLES
# Create publication/report-ready tables from Phase 1-4 outputs
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 220)

# ============================================================
# PATHS
# ============================================================

import os
from pathlib import Path
import pandas as pd
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Phase 3
PHASE3_DIR = PROJECT_DIR / "model" / "phase3_multimodal_integration"

PHASE3_1_DIR = PHASE3_DIR / "phase3_1_modelling_light"
PHASE3_1_RESULT_DIR = PHASE3_1_DIR / "results"

PHASE3_2_DIR = PHASE3_DIR / "phase3_2_error_modality_contribution"
PHASE3_2_RESULT_DIR = PHASE3_2_DIR / "results"

# Phase 4.0
PHASE4_DIR = PROJECT_DIR / "model" / "phase4_biological_validation_preparation"
PHASE4_RESULT_DIR = PHASE4_DIR / "results"
PHASE4_GENE_LIST_DIR = PHASE4_DIR / "gene_lists"

# Phase 4.1
PHASE4_1_DIR = PROJECT_DIR / "model" / "phase4_1_biological_validation_enrichment"
PHASE4_1_RESULT_DIR = PHASE4_1_DIR / "results"
PHASE4_1_ENRICHMENT_DIR = PHASE4_1_DIR / "enrichment_results"

# Final report table output
FINAL_TABLE_DIR = PROJECT_DIR / "model" / "final_report_tables"
FINAL_CSV_DIR = FINAL_TABLE_DIR / "csv"
FINAL_MD_DIR = FINAL_TABLE_DIR / "markdown"
FINAL_EXCEL_DIR = FINAL_TABLE_DIR / "excel"

for folder in [FINAL_TABLE_DIR, FINAL_CSV_DIR, FINAL_MD_DIR, FINAL_EXCEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Final report table output:", FINAL_TABLE_DIR)

Mounted at /content/drive
Final report table output: /content/drive/MyDrive/Project_Protein/model/final_report_tables


In [2]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def read_csv_if_exists(path, required=True):
    path = Path(path)
    if path.exists():
        print("Loaded:", path)
        return pd.read_csv(path)
    else:
        msg = f"Missing file: {path}"
        if required:
            raise FileNotFoundError(msg)
        print("Warning:", msg)
        return pd.DataFrame()


def save_table(df, table_name, index=False):
    """
    Save one table as CSV and Markdown.
    """
    csv_path = FINAL_CSV_DIR / f"{table_name}.csv"
    md_path = FINAL_MD_DIR / f"{table_name}.md"

    df.to_csv(csv_path, index=index)

    with open(md_path, "w") as f:
        f.write(df.to_markdown(index=index))

    print(f"Saved CSV: {csv_path}")
    print(f"Saved MD:  {md_path}")

    return csv_path, md_path


def round_numeric_columns(df, digits=4):
    df = df.copy()

    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].round(digits)

    return df


def clean_gene_list_text(x, max_genes=None):
    if pd.isna(x):
        return ""

    genes = str(x).split(";")
    genes = [g.strip() for g in genes if g.strip() != ""]

    if max_genes is not None:
        genes = genes[:max_genes]

    return "; ".join(genes)


def first_non_missing(row, cols):
    for col in cols:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip() != "":
            return row[col]
    return None

In [3]:
# ============================================================
# LOAD SOURCE TABLES
# ============================================================

# Phase 4.0 final master comparison
master_comparison_path = PHASE4_RESULT_DIR / "phase4_0A_final_master_comparison_table.csv"
master_comparison_df = read_csv_if_exists(master_comparison_path)

# Phase 4.0 official predictions / top 50 / candidate table
official_predictions_path = PHASE4_RESULT_DIR / "phase4_0B_official_combined_svm_all_test_predictions.csv"
candidate_table_path = PHASE4_RESULT_DIR / "phase4_0B_biological_validation_candidate_table.csv"
official_metric_path = PHASE4_RESULT_DIR / "phase4_0B_official_combined_svm_metric_check.csv"

official_predictions_df = read_csv_if_exists(official_predictions_path)
candidate_table_df = read_csv_if_exists(candidate_table_path)
official_metric_df = read_csv_if_exists(official_metric_path)

# Phase 4.1 enrichment / validation outputs
enrichment_summary_path = PHASE4_1_RESULT_DIR / "phase4_1_enrichment_summary_by_gene_list.csv"
t2d_relevant_terms_path = PHASE4_1_RESULT_DIR / "phase4_1_t2d_relevant_enrichment_terms.csv"
curated_overlap_path = PHASE4_1_RESULT_DIR / "phase4_1_curated_t2d_gene_set_overlap.csv"
top50_validation_path = PHASE4_1_RESULT_DIR / "phase4_1_top50_gene_curated_validation_status.csv"
top50_support_summary_path = PHASE4_1_RESULT_DIR / "phase4_1_top50_curated_support_summary.csv"
priority_table_path = PHASE4_1_RESULT_DIR / "phase4_1_top50_biological_validation_priority_table.csv"

enrichment_summary_df = read_csv_if_exists(enrichment_summary_path)
t2d_relevant_terms_df = read_csv_if_exists(t2d_relevant_terms_path)
curated_overlap_df = read_csv_if_exists(curated_overlap_path)
top50_validation_df = read_csv_if_exists(top50_validation_path)
top50_support_summary_df = read_csv_if_exists(top50_support_summary_path)
priority_table_df = read_csv_if_exists(priority_table_path)

# Phase 3.2 optional files
error_group_counts_path = PHASE3_2_RESULT_DIR / "phase3_2A_error_group_counts.csv"
block_permutation_path = PHASE3_2_RESULT_DIR / "phase3_2B_block_permutation_importance_summary.csv"
lr_modality_path = PHASE3_2_RESULT_DIR / "phase3_2C_lr_modality_coefficient_summary.csv"
rescued_path = PHASE4_GENE_LIST_DIR / "phase4_0B_combined_rescued_genes.csv"

error_group_counts_df = read_csv_if_exists(error_group_counts_path, required=False)
block_permutation_df = read_csv_if_exists(block_permutation_path, required=False)
lr_modality_df = read_csv_if_exists(lr_modality_path, required=False)
combined_rescued_df = read_csv_if_exists(rescued_path, required=False)

print("\nLoaded source tables complete.")

Loaded: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0A_final_master_comparison_table.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_official_combined_svm_all_test_predictions.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_biological_validation_candidate_table.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase4_biological_validation_preparation/results/phase4_0B_official_combined_svm_metric_check.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase4_1_biological_validation_enrichment/results/phase4_1_enrichment_summary_by_gene_list.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase4_1_biological_validation_enrichment/results/phase4_1_t2d_relevant_enrichment_terms.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase4_1_biological_validation_enrichment/r

In [4]:
# ============================================================
# TABLE 1 — FINAL MODEL COMPARISON
# ============================================================

table1_cols = [
    "phase",
    "branch",
    "representation",
    "embedding_policy",
    "feature_type",
    "model",
    "selection_policy",
    "threshold_policy",
    "test_roc_auc",
    "test_pr_auc",
    "test_f1",
    "test_mcc",
    "main_note"
]

table1_final_model_comparison = master_comparison_df[
    [col for col in table1_cols if col in master_comparison_df.columns]
].copy()

table1_final_model_comparison = table1_final_model_comparison.sort_values(
    by=["test_roc_auc", "test_pr_auc", "test_mcc"],
    ascending=False
).reset_index(drop=True)

table1_final_model_comparison.insert(
    0,
    "rank_by_roc_auc",
    np.arange(1, len(table1_final_model_comparison) + 1)
)

table1_final_model_comparison = round_numeric_columns(table1_final_model_comparison, 4)

display(table1_final_model_comparison)

save_table(
    table1_final_model_comparison,
    "table1_final_model_comparison"
)

,rank_by_roc_auc,phase,branch,representation,embedding_policy,feature_type,model,selection_policy,threshold_policy,test_roc_auc,test_pr_auc,test_f1,test_mcc,main_note
0,1,3.1,Multimodal,ProtBERT SW + K3/K4/Basic,early fusion direct concatenation,protein embedding + genomic handcrafted,SVM RBF,validation ROC-AUC,default_0.5,0.7290,0.7573,0.6590,0.3438,Official multimodal integration model
1,2,3.1,Protein shared split,ProtBERT SW,shared split,protein foundation embedding,SVM RBF,validation ROC-AUC,default_0.5,0.7274,0.7433,0.6667,0.3215,Best protein-only model on shared Phase 3 split
2,3,2.2,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,SVM RBF,diagnostic test comparison,default_0.5,0.6765,0.6504,0.6620,0.2933,"Diagnostic best genomic model, not official selection"
3,4,2.1,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,Random Forest,validation ROC-AUC,default_0.5,0.6496,0.6327,0.6406,0.2557,Official genomic handcrafted model
4,5,1.2E,Protein,ProtBERT,sliding_window_1022_stride_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6487,0.6551,0.5896,0.1941,Best protein ranking representation before shared split
5,6,1.2D,Protein,ProtBERT,truncated_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6371,0.6423,0.6014,0.1943,ProtBERT truncated
6,7,1.2B,Protein,ESM2_t6_8M,sliding_window_1022_stride_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.6277,0.6278,0.5870,0.1650,ESM-2 small sliding-window
7,8,1.2A,Protein,ESM2_t6_8M,truncated_1022,protein foundation embedding,Stacking,validation ROC-AUC,default_0.5,0.6202,0.6188,0.5926,0.1941,ESM-2 small truncated baseline
8,9,1.2C,Protein,ESM2_t12_35M,truncated_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.5875,0.5942,0.5654,0.0995,Larger ESM-2 did not improve under truncation
9,10,1.1,Protein,AAC + Physchem,handcrafted,handcrafted protein descriptors,Random Forest,validation ROC-AUC,default_0.5,0.5520,0.5550,0.5390,0.0480,Protein handcrafted baseline


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table1_final_model_comparison.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table1_final_model_comparison.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table1_final_model_comparison.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table1_final_model_comparison.md'))

In [5]:
# ============================================================
# TABLE 2 — OFFICIAL MODEL PERFORMANCE
# ============================================================

table2_official_model_performance = official_metric_df.copy()

# Keep clean columns
table2_cols = [
    "model",
    "threshold",
    "accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc",
    "mcc",
    "tn",
    "fp",
    "fn",
    "tp"
]

table2_official_model_performance = table2_official_model_performance[
    [col for col in table2_cols if col in table2_official_model_performance.columns]
].copy()

table2_official_model_performance = round_numeric_columns(
    table2_official_model_performance,
    4
)

display(table2_official_model_performance)

save_table(
    table2_official_model_performance,
    "table2_official_model_performance"
)

,model,threshold,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Official Combined Protein+Genomic SVM RBF,0.5,0.6716,0.6825,0.637,0.7059,0.659,0.729,0.7573,0.3438,96,40,49,86


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table2_official_model_performance.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table2_official_model_performance.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table2_official_model_performance.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table2_official_model_performance.md'))

In [6]:
# ============================================================
# TABLE 3A — ERROR GROUP SUMMARY
# ============================================================

if not error_group_counts_df.empty:
    table3a_error_group_summary = error_group_counts_df.copy()

    table3a_error_group_summary = table3a_error_group_summary.rename(columns={
        "integration_error_group": "error_group",
        "n": "n_genes",
        "percentage": "percentage_of_test_set"
    })

    table3a_error_group_summary = round_numeric_columns(table3a_error_group_summary, 2)

    display(table3a_error_group_summary)

    save_table(
        table3a_error_group_summary,
        "table3a_error_group_summary"
    )
else:
    print("Skipping Table 3A because error_group_counts_df is missing.")


# ============================================================
# TABLE 3B — BLOCK PERMUTATION IMPORTANCE
# ============================================================

if not block_permutation_df.empty:
    table3b_block_permutation = block_permutation_df.copy()

    keep_cols = [
        "permuted_block",
        "n_repeats",
        "baseline_roc_auc",
        "mean_roc_auc_after_permutation",
        "mean_roc_auc_drop",
        "std_roc_auc_drop",
        "baseline_pr_auc",
        "mean_pr_auc_after_permutation",
        "mean_pr_auc_drop",
        "std_pr_auc_drop",
        "baseline_f1",
        "mean_f1_after_permutation",
        "mean_f1_drop",
        "baseline_mcc",
        "mean_mcc_after_permutation",
        "mean_mcc_drop"
    ]

    table3b_block_permutation = table3b_block_permutation[
        [col for col in keep_cols if col in table3b_block_permutation.columns]
    ].copy()

    table3b_block_permutation = round_numeric_columns(table3b_block_permutation, 4)

    display(table3b_block_permutation)

    save_table(
        table3b_block_permutation,
        "table3b_block_permutation_importance"
    )
else:
    print("Skipping Table 3B because block_permutation_df is missing.")


# ============================================================
# TABLE 3C — LOGISTIC REGRESSION MODALITY COEFFICIENT SHARE
# ============================================================

if not lr_modality_df.empty:
    table3c_lr_modality_contribution = lr_modality_df.copy()

    keep_cols = [
        "modality",
        "n_features",
        "sum_abs_coefficient",
        "mean_abs_coefficient",
        "median_abs_coefficient",
        "max_abs_coefficient",
        "share_of_total_abs_coefficient"
    ]

    table3c_lr_modality_contribution = table3c_lr_modality_contribution[
        [col for col in keep_cols if col in table3c_lr_modality_contribution.columns]
    ].copy()

    table3c_lr_modality_contribution = round_numeric_columns(
        table3c_lr_modality_contribution,
        4
    )

    display(table3c_lr_modality_contribution)

    save_table(
        table3c_lr_modality_contribution,
        "table3c_lr_modality_contribution"
    )
else:
    print("Skipping Table 3C because lr_modality_df is missing.")

,error_group,n_genes,percentage_of_test_set
0,both_correct,164,60.52
1,both_wrong,74,27.31
2,combined_correct_protein_wrong,18,6.64
3,protein_correct_combined_wrong,15,5.54


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table3a_error_group_summary.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table3a_error_group_summary.md


,permuted_block,n_repeats,baseline_roc_auc,mean_roc_auc_after_permutation,mean_roc_auc_drop,std_roc_auc_drop,baseline_pr_auc,mean_pr_auc_after_permutation,mean_pr_auc_drop,std_pr_auc_drop,baseline_f1,mean_f1_after_permutation,mean_f1_drop,baseline_mcc,mean_mcc_after_permutation,mean_mcc_drop
0,protein_block,30,0.729,0.5301,0.1989,0.0396,0.7573,0.5330,0.2243,0.0316,0.659,0.5113,0.1477,0.3438,0.0354,0.3084
1,genomic_block,30,0.729,0.7069,0.0221,0.0157,0.7573,0.7178,0.0394,0.0232,0.659,0.6516,0.0074,0.3438,0.3146,0.0292


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table3b_block_permutation_importance.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table3b_block_permutation_importance.md


,modality,n_features,sum_abs_coefficient,mean_abs_coefficient,median_abs_coefficient,max_abs_coefficient,share_of_total_abs_coefficient
0,genomic,355,2.7735,0.0078,0.0064,0.0365,0.2954
1,protein,1024,6.6155,0.0065,0.0054,0.0317,0.7046


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table3c_lr_modality_contribution.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table3c_lr_modality_contribution.md


In [7]:
# ============================================================
# TABLE 4 — ENRICHMENT SUMMARY BY GENE LIST
# ============================================================

table4_enrichment_summary = enrichment_summary_df.copy()

table4_cols = [
    "gene_list_name",
    "n_input_genes",
    "n_significant_terms",
    "n_t2d_relevant_terms",
    "best_p_value",
    "top_term",
    "top_t2d_relevant_term"
]

table4_enrichment_summary = table4_enrichment_summary[
    [col for col in table4_cols if col in table4_enrichment_summary.columns]
].copy()

table4_enrichment_summary = table4_enrichment_summary.sort_values(
    by=["n_t2d_relevant_terms", "n_significant_terms"],
    ascending=False
).reset_index(drop=True)

table4_enrichment_summary["best_p_value_scientific"] = table4_enrichment_summary["best_p_value"].apply(
    lambda x: f"{x:.2e}" if pd.notna(x) else ""
)

table4_enrichment_summary = table4_enrichment_summary[
    [
        "gene_list_name",
        "n_input_genes",
        "n_significant_terms",
        "n_t2d_relevant_terms",
        "best_p_value_scientific",
        "top_term",
        "top_t2d_relevant_term"
    ]
]

display(table4_enrichment_summary)

save_table(
    table4_enrichment_summary,
    "table4_enrichment_summary_by_gene_list"
)

,gene_list_name,n_input_genes,n_significant_terms,n_t2d_relevant_terms,best_p_value_scientific,top_term,top_t2d_relevant_term
0,top_true_positives,86,175,63,6.97e-11,respiratory chain complex I,oxidative phosphorylation
1,top_50_combined_predictions,50,102,41,1.85e-05,NADH dehydrogenase (ubiquinone) activity,Oxidative phosphorylation
2,high_confidence_false_negatives,49,3,0,4.80e-02,Anterior uveitis,NaN
3,both_wrong_genes,74,1,0,1.07e-02,regulation of toll-like receptor 3 signaling pathway,NaN
4,high_confidence_false_positives,40,0,0,,NaN,NaN
5,combined_rescued_genes,18,0,0,,NaN,NaN


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table4_enrichment_summary_by_gene_list.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table4_enrichment_summary_by_gene_list.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table4_enrichment_summary_by_gene_list.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table4_enrichment_summary_by_gene_list.md'))

In [8]:
# ============================================================
# TABLE 5 — TOP T2D-RELEVANT ENRICHMENT TERMS
# ============================================================

table5 = t2d_relevant_terms_df.copy()

# Prioritize top true positives and top 50 combined predictions
priority_gene_lists = [
    "top_true_positives",
    "top_50_combined_predictions"
]

table5 = table5[
    table5["gene_list_name"].isin(priority_gene_lists)
].copy()

# Keep useful columns
table5_cols = [
    "gene_list_name",
    "source",
    "native",
    "name",
    "p_value",
    "minus_log10_p_value",
    "intersection_size",
    "term_size",
    "precision",
    "recall",
    "intersections",
    "t2d_relevance_score",
    "matched_biological_groups"
]

table5 = table5[
    [col for col in table5_cols if col in table5.columns]
].copy()

# Convert p-values and intersections
table5["p_value_scientific"] = table5["p_value"].apply(
    lambda x: f"{x:.2e}" if pd.notna(x) else ""
)

if "intersections" in table5.columns:
    table5["overlap_genes"] = table5["intersections"].apply(
        lambda x: clean_gene_list_text(str(x).replace("[", "").replace("]", "").replace("'", "").replace(",", ";"), max_genes=20)
    )
else:
    table5["overlap_genes"] = ""

# Sort by gene list, relevance, p-value
table5 = table5.sort_values(
    by=["gene_list_name", "t2d_relevance_score", "p_value"],
    ascending=[True, False, True]
).reset_index(drop=True)

# Keep top terms per list
TOP_TERMS_PER_LIST = 15

table5_top_t2d_terms = (
    table5
    .groupby("gene_list_name", group_keys=False)
    .head(TOP_TERMS_PER_LIST)
    .reset_index(drop=True)
)

table5_final_cols = [
    "gene_list_name",
    "source",
    "native",
    "name",
    "p_value_scientific",
    "intersection_size",
    "term_size",
    "precision",
    "recall",
    "overlap_genes",
    "matched_biological_groups"
]

table5_top_t2d_terms = table5_top_t2d_terms[
    [col for col in table5_final_cols if col in table5_top_t2d_terms.columns]
].copy()

table5_top_t2d_terms = round_numeric_columns(table5_top_t2d_terms, 4)

display(table5_top_t2d_terms)

save_table(
    table5_top_t2d_terms,
    "table5_top_t2d_relevant_enrichment_terms"
)

,gene_list_name,source,native,name,p_value_scientific,intersection_size,term_size,precision,recall,overlap_genes,matched_biological_groups
0,top_50_combined_predictions,KEGG,KEGG:00190,Oxidative phosphorylation,1.29e-02,5,136,0.1471,0.0368,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy;signaling_regulation
1,top_50_combined_predictions,WP,WP:WP623,Oxidative phosphorylation,1.48e-02,4,60,0.1111,0.0667,NDUFB2; NDUFV1; NDUFA7; NDUFB1,mitochondria_energy;signaling_regulation
2,top_50_combined_predictions,GO:BP,GO:0006119,oxidative phosphorylation,3.08e-02,5,146,0.1064,0.0342,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy;signaling_regulation
3,top_50_combined_predictions,GO:BP,GO:0031016,pancreas development,4.90e-02,4,79,0.0851,0.0506,NEUROD1; INSR; ISL1; PDX1,development_beta_cell;diabetes_glucose_insulin
4,top_50_combined_predictions,GO:MF,GO:0008137,NADH dehydrogenase (ubiquinone) activity,1.85e-05,5,41,0.1042,0.1220,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
5,top_50_combined_predictions,GO:CC,GO:0030964,NADH dehydrogenase complex,2.14e-05,5,49,0.1000,0.1020,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
6,top_50_combined_predictions,GO:CC,GO:0045271,respiratory chain complex I,2.14e-05,5,49,0.1000,0.1020,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
7,top_50_combined_predictions,GO:BP,GO:0006120,"mitochondrial electron transport, NADH to ubiquinone",2.00e-04,5,53,0.1064,0.0943,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
8,top_50_combined_predictions,GO:MF,GO:0001228,"DNA-binding transcription activator activity, RNA polymerase II-specific",2.91e-04,9,431,0.1875,0.0209,ETV1; NEUROD1; HHEX; TCF12; BCL11B; MEIS1; NFAT5; ISL1; TFAP2B,signaling_regulation
9,top_50_combined_predictions,GO:MF,GO:0001216,DNA-binding transcription activator activity,3.26e-04,9,437,0.1875,0.0206,ETV1; NEUROD1; HHEX; TCF12; BCL11B; MEIS1; NFAT5; ISL1; TFAP2B,signaling_regulation


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table5_top_t2d_relevant_enrichment_terms.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table5_top_t2d_relevant_enrichment_terms.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table5_top_t2d_relevant_enrichment_terms.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table5_top_t2d_relevant_enrichment_terms.md'))

In [9]:
# ============================================================
# TABLE 6 — CURATED T2D GENE SET OVERLAP
# ============================================================

table6_curated_overlap = curated_overlap_df.copy()

# Keep meaningful rows only: overlap > 0
table6_curated_overlap = table6_curated_overlap[
    table6_curated_overlap["n_overlap"] > 0
].copy()

table6_cols = [
    "query_gene_list",
    "curated_set",
    "n_query_genes",
    "n_curated_genes",
    "n_overlap",
    "overlap_fraction_query",
    "overlap_genes"
]

table6_curated_overlap = table6_curated_overlap[
    [col for col in table6_cols if col in table6_curated_overlap.columns]
].copy()

table6_curated_overlap["overlap_genes"] = table6_curated_overlap["overlap_genes"].apply(
    lambda x: clean_gene_list_text(x)
)

table6_curated_overlap = table6_curated_overlap.sort_values(
    by=["n_overlap", "overlap_fraction_query"],
    ascending=False
).reset_index(drop=True)

table6_curated_overlap = round_numeric_columns(table6_curated_overlap, 4)

display(table6_curated_overlap)

save_table(
    table6_curated_overlap,
    "table6_curated_t2d_gene_set_overlap"
)

,query_gene_list,curated_set,n_query_genes,n_curated_genes,n_overlap,overlap_fraction_query,overlap_genes
0,top_true_positives,known_T2D_GWAS_or_monogenic_related,86,28,8,0.0930,ABCC8; HHEX; IGF2BP2; INSR; KCNJ11; MTNR1B; NEUROD1; PDX1
1,top_true_positives,insulin_secretion_beta_cell,86,19,8,0.0930,ABCC8; CACNA1D; DPP4; GAD1; INSR; KCNJ11; NEUROD1; PDX1
2,top_50_combined_predictions,insulin_secretion_beta_cell,50,19,6,0.1200,CACNA1D; DPP4; GAD1; INSR; NEUROD1; PDX1
3,top_true_positives,mitochondrial_oxphos,86,17,6,0.0698,NDUFA2; NDUFA7; NDUFB1; NDUFB2; NDUFB3; NDUFV1
4,top_50_combined_predictions,known_T2D_GWAS_or_monogenic_related,50,28,5,0.1000,HHEX; IGF2BP2; INSR; NEUROD1; PDX1
5,top_50_combined_predictions,mitochondrial_oxphos,50,17,5,0.1000,NDUFA7; NDUFB1; NDUFB2; NDUFB3; NDUFV1
6,combined_rescued_genes,known_T2D_GWAS_or_monogenic_related,18,28,2,0.1111,ABCC8; MTNR1B
7,high_confidence_false_negatives,known_T2D_GWAS_or_monogenic_related,49,28,2,0.0408,PAX4; THADA
8,both_wrong_genes,known_T2D_GWAS_or_monogenic_related,74,28,2,0.0270,PAX4; THADA
9,combined_rescued_genes,insulin_secretion_beta_cell,18,19,1,0.0556,ABCC8


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table6_curated_t2d_gene_set_overlap.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table6_curated_t2d_gene_set_overlap.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table6_curated_t2d_gene_set_overlap.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table6_curated_t2d_gene_set_overlap.md'))

In [10]:
# ============================================================
# TABLE 7 — TOP 50 COMBINED PREDICTIONS WITH CURATED SUPPORT
# ============================================================

table7_top50_genes = top50_validation_df.copy()

table7_cols = [
    "rank",
    "gene_id",
    "gene_symbol",
    "true_label",
    "combined_svm_score",
    "combined_error_type",
    "has_curated_t2d_support",
    "curated_t2d_related_sets"
]

table7_top50_genes = table7_top50_genes[
    [col for col in table7_cols if col in table7_top50_genes.columns]
].copy()

table7_top50_genes = table7_top50_genes.sort_values(
    by="rank",
    ascending=True
).reset_index(drop=True)

table7_top50_genes = round_numeric_columns(table7_top50_genes, 4)

display(table7_top50_genes)

save_table(
    table7_top50_genes,
    "table7_top50_combined_predictions_curated_support"
)

,rank,gene_id,gene_symbol,true_label,combined_svm_score,combined_error_type,has_curated_t2d_support,curated_t2d_related_sets
0,1,ENSG00000053254,FOXN3,1,0.8846,TP,False,NaN
1,2,ENSG00000276644,DACH1,1,0.8412,TP,False,NaN
2,3,ENSG00000090266,NDUFB2,1,0.8312,TP,True,mitochondrial_oxphos
3,4,ENSG00000006468,ETV1,1,0.8241,TP,False,NaN
4,5,ENSG00000162992,NEUROD1,1,0.8185,TP,True,known_T2D_GWAS_or_monogenic_related;insulin_secretion_beta_cell
5,6,ENSG00000171105,INSR,1,0.8175,TP,True,known_T2D_GWAS_or_monogenic_related;insulin_secretion_beta_cell
6,7,ENSG00000128683,GAD1,1,0.8134,TP,True,insulin_secretion_beta_cell
7,8,ENSG00000107864,CPEB3,1,0.8088,TP,False,NaN
8,9,ENSG00000121297,TSHZ3,1,0.8087,TP,False,NaN
9,10,ENSG00000127334,DYRK2,1,0.8058,TP,False,NaN


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table7_top50_combined_predictions_curated_support.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table7_top50_combined_predictions_curated_support.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table7_top50_combined_predictions_curated_support.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table7_top50_combined_predictions_curated_support.md'))

In [11]:
# ============================================================
# TABLE 8 — TOP 50 CURATED SUPPORT SUMMARY
# ============================================================

table8_top50_support_summary = top50_support_summary_df.copy()

table8_cols = [
    "n_top50_genes",
    "n_true_positive",
    "n_false_positive",
    "n_with_curated_t2d_support",
    "pct_with_curated_t2d_support",
    "supported_genes"
]

table8_top50_support_summary = table8_top50_support_summary[
    [col for col in table8_cols if col in table8_top50_support_summary.columns]
].copy()

table8_top50_support_summary["supported_genes"] = table8_top50_support_summary["supported_genes"].apply(
    lambda x: clean_gene_list_text(x)
)

table8_top50_support_summary = round_numeric_columns(table8_top50_support_summary, 2)

display(table8_top50_support_summary)

save_table(
    table8_top50_support_summary,
    "table8_top50_curated_support_summary"
)

,n_top50_genes,n_true_positive,n_false_positive,n_with_curated_t2d_support,pct_with_curated_t2d_support,supported_genes
0,50,43,7,15,30.0,NDUFB2; NEUROD1; INSR; GAD1; NDUFV1; CACNA1D; HHEX; NDUFB3; SLC5A1; IGF2BP2; NDUFA7; SCD5; NDUFB1; DPP4; PDX1


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table8_top50_curated_support_summary.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table8_top50_curated_support_summary.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table8_top50_curated_support_summary.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table8_top50_curated_support_summary.md'))

In [12]:
# ============================================================
# TABLE 9 — COMBINED-RESCUED GENES WITH BIOLOGICAL NOTE
# ============================================================

if not combined_rescued_df.empty:
    table9_rescued_genes = combined_rescued_df.copy()

    # Add known curated tags manually from your curated overlap results
    known_t2d_rescued = {"ABCC8", "MTNR1B"}
    insulin_beta_cell = {"ABCC8"}

    def rescued_gene_note(row):
        gene = str(row.get("gene_symbol", "")).upper()

        notes = []

        if gene in known_t2d_rescued:
            notes.append("known_T2D_or_GWAS_related")
        if gene in insulin_beta_cell:
            notes.append("insulin_secretion_beta_cell_related")

        if len(notes) == 0:
            notes.append("no_curated_support_in_current_gene_sets")

        return "; ".join(notes)

    table9_rescued_genes["biological_note"] = table9_rescued_genes.apply(
        rescued_gene_note,
        axis=1
    )

    table9_cols = [
        "gene_id",
        "gene_symbol",
        "true_label",
        "protein_svm_score",
        "combined_svm_score",
        "protein_svm_pred",
        "combined_svm_pred",
        "protein_error_type",
        "combined_error_type",
        "score_delta_combined_minus_protein",
        "biological_note"
    ]

    table9_rescued_genes = table9_rescued_genes[
        [col for col in table9_cols if col in table9_rescued_genes.columns]
    ].copy()

    table9_rescued_genes = table9_rescued_genes.sort_values(
        by=["true_label", "combined_svm_score"],
        ascending=[False, False]
    ).reset_index(drop=True)

    table9_rescued_genes = round_numeric_columns(table9_rescued_genes, 4)

    display(table9_rescued_genes)

    save_table(
        table9_rescued_genes,
        "table9_combined_rescued_genes_biological_notes"
    )

else:
    print("Skipping Table 9 because combined_rescued_df is missing.")

,gene_id,gene_symbol,true_label,protein_svm_score,combined_svm_score,protein_svm_pred,combined_svm_pred,protein_error_type,combined_error_type,score_delta_combined_minus_protein,biological_note
0,ENSG00000151883,PARP8,1,0.4571,0.5407,0,1,FN,TP,0.0837,no_curated_support_in_current_gene_sets
1,ENSG00000006071,ABCC8,1,0.4690,0.5225,0,1,FN,TP,0.0535,known_T2D_or_GWAS_related; insulin_secretion_beta_cell_related
2,ENSG00000134640,MTNR1B,1,0.4946,0.5000,0,1,FN,TP,0.0054,known_T2D_or_GWAS_related
3,ENSG00000103978,TMEM87A,0,0.6179,0.4942,1,0,FP,TN,-0.1236,no_curated_support_in_current_gene_sets
4,ENSG00000122550,KLHL7,0,0.7143,0.4918,1,0,FP,TN,-0.2226,no_curated_support_in_current_gene_sets
5,ENSG00000105738,SIPA1L3,0,0.5978,0.4869,1,0,FP,TN,-0.1109,no_curated_support_in_current_gene_sets
6,ENSG00000077327,SPAG6,0,0.5563,0.4764,1,0,FP,TN,-0.0799,no_curated_support_in_current_gene_sets
7,ENSG00000197530,MIB2,0,0.5765,0.4636,1,0,FP,TN,-0.1129,no_curated_support_in_current_gene_sets
8,ENSG00000154832,CXXC1,0,0.6072,0.4602,1,0,FP,TN,-0.1469,no_curated_support_in_current_gene_sets
9,ENSG00000173320,STOX2,0,0.5691,0.4526,1,0,FP,TN,-0.1166,no_curated_support_in_current_gene_sets


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table9_combined_rescued_genes_biological_notes.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table9_combined_rescued_genes_biological_notes.md


In [13]:
# ============================================================
# TABLE 10 — COMPACT FINAL RESULTS TABLE
# ============================================================

compact_records = []

# Official model from table 2
if not table2_official_model_performance.empty:
    official_row = table2_official_model_performance.iloc[0].to_dict()

    compact_records.append({
        "section": "Model performance",
        "key_result": "Official model",
        "value": official_row.get("model", "Combined Protein+Genomic SVM RBF"),
        "interpretation": "Selected multimodal model for final evaluation"
    })

    compact_records.append({
        "section": "Model performance",
        "key_result": "ROC-AUC / PR-AUC / F1 / MCC",
        "value": f"{official_row.get('roc_auc', np.nan):.4f} / {official_row.get('pr_auc', np.nan):.4f} / {official_row.get('f1', np.nan):.4f} / {official_row.get('mcc', np.nan):.4f}",
        "interpretation": "Overall held-out test performance"
    })

# Enrichment summary
top_tp_row = enrichment_summary_df[
    enrichment_summary_df["gene_list_name"] == "top_true_positives"
]

if not top_tp_row.empty:
    r = top_tp_row.iloc[0]

    compact_records.append({
        "section": "Biological validation",
        "key_result": "Top true positives enrichment",
        "value": f"{int(r['n_significant_terms'])} significant terms; {int(r['n_t2d_relevant_terms'])} T2D-relevant terms",
        "interpretation": "Correct model predictions are biologically enriched"
    })

top50_row = enrichment_summary_df[
    enrichment_summary_df["gene_list_name"] == "top_50_combined_predictions"
]

if not top50_row.empty:
    r = top50_row.iloc[0]

    compact_records.append({
        "section": "Biological validation",
        "key_result": "Top 50 predictions enrichment",
        "value": f"{int(r['n_significant_terms'])} significant terms; {int(r['n_t2d_relevant_terms'])} T2D-relevant terms",
        "interpretation": "Top-ranked predictions show pathway-level biological signal"
    })

# Curated support
if not top50_support_summary_df.empty:
    r = top50_support_summary_df.iloc[0]

    compact_records.append({
        "section": "Curated support",
        "key_result": "Top 50 curated T2D support",
        "value": f"{int(r['n_with_curated_t2d_support'])}/{int(r['n_top50_genes'])} genes ({float(r['pct_with_curated_t2d_support']):.1f}%)",
        "interpretation": "Top-ranked genes overlap with curated T2D/metabolic gene sets"
    })

# Rescued genes
if not combined_rescued_df.empty:
    compact_records.append({
        "section": "Multimodal integration",
        "key_result": "Biologically supported rescued genes",
        "value": "ABCC8; MTNR1B",
        "interpretation": "Some genes corrected by multimodal integration have known diabetes relevance"
    })

table10_compact_final_results = pd.DataFrame(compact_records)

display(table10_compact_final_results)

save_table(
    table10_compact_final_results,
    "table10_compact_final_results_summary"
)

,section,key_result,value,interpretation
0,Model performance,Official model,Official Combined Protein+Genomic SVM RBF,Selected multimodal model for final evaluation
1,Model performance,ROC-AUC / PR-AUC / F1 / MCC,0.7290 / 0.7573 / 0.6590 / 0.3438,Overall held-out test performance
2,Biological validation,Top true positives enrichment,175 significant terms; 63 T2D-relevant terms,Correct model predictions are biologically enriched
3,Biological validation,Top 50 predictions enrichment,102 significant terms; 41 T2D-relevant terms,Top-ranked predictions show pathway-level biological signal
4,Curated support,Top 50 curated T2D support,15/50 genes (30.0%),Top-ranked genes overlap with curated T2D/metabolic gene sets
5,Multimodal integration,Biologically supported rescued genes,ABCC8; MTNR1B,Some genes corrected by multimodal integration have known diabetes relevance


Saved CSV: /content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table10_compact_final_results_summary.csv
Saved MD:  /content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table10_compact_final_results_summary.md


(PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table10_compact_final_results_summary.csv'),
 PosixPath('/content/drive/MyDrive/Project_Protein/model/final_report_tables/markdown/table10_compact_final_results_summary.md'))

In [14]:
# ============================================================
# EXPORT ALL TABLES TO ONE EXCEL WORKBOOK
# ============================================================

excel_path = FINAL_EXCEL_DIR / "final_report_tables.xlsx"

tables_to_export = {
    "T1_Model_Comparison": table1_final_model_comparison,
    "T2_Official_Model": table2_official_model_performance,
    "T4_Enrichment_Summary": table4_enrichment_summary,
    "T5_T2D_Enrichment": table5_top_t2d_terms,
    "T6_Curated_Overlap": table6_curated_overlap,
    "T7_Top50_Genes": table7_top50_genes,
    "T8_Top50_Support": table8_top50_support_summary,
    "T10_Compact_Results": table10_compact_final_results,
}

if "table3a_error_group_summary" in globals():
    tables_to_export["T3A_Error_Groups"] = table3a_error_group_summary

if "table3b_block_permutation" in globals():
    tables_to_export["T3B_Block_Permutation"] = table3b_block_permutation

if "table3c_lr_modality_contribution" in globals():
    tables_to_export["T3C_Modality_Contribution"] = table3c_lr_modality_contribution

if "table9_rescued_genes" in globals():
    tables_to_export["T9_Rescued_Genes"] = table9_rescued_genes

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, df in tables_to_export.items():
        safe_sheet_name = sheet_name[:31]
        df.to_excel(writer, sheet_name=safe_sheet_name, index=False)

print("Saved Excel workbook:", excel_path)

Saved Excel workbook: /content/drive/MyDrive/Project_Protein/model/final_report_tables/excel/final_report_tables.xlsx


In [15]:
# ============================================================
# LIST FINAL OUTPUTS
# ============================================================

print("=== CSV TABLES ===")
for p in sorted(FINAL_CSV_DIR.glob("*.csv")):
    print(p)

print("\n=== MARKDOWN TABLES ===")
for p in sorted(FINAL_MD_DIR.glob("*.md")):
    print(p)

print("\n=== EXCEL WORKBOOK ===")
for p in sorted(FINAL_EXCEL_DIR.glob("*.xlsx")):
    print(p)

=== CSV TABLES ===
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table10_compact_final_results_summary.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table1_final_model_comparison.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table2_official_model_performance.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table3a_error_group_summary.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table3b_block_permutation_importance.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table3c_lr_modality_contribution.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table4_enrichment_summary_by_gene_list.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table5_top_t2d_relevant_enrichment_terms.csv
/content/drive/MyDrive/Project_Protein/model/final_report_tables/csv/table6_curated_t2d_gene_set_overlap.csv
/content

In [16]:
display(table1_final_model_comparison)
display(table4_enrichment_summary)
display(table5_top_t2d_terms)
display(table6_curated_overlap)
display(table8_top50_support_summary)
display(table10_compact_final_results)

,rank_by_roc_auc,phase,branch,representation,embedding_policy,feature_type,model,selection_policy,threshold_policy,test_roc_auc,test_pr_auc,test_f1,test_mcc,main_note
0,1,3.1,Multimodal,ProtBERT SW + K3/K4/Basic,early fusion direct concatenation,protein embedding + genomic handcrafted,SVM RBF,validation ROC-AUC,default_0.5,0.7290,0.7573,0.6590,0.3438,Official multimodal integration model
1,2,3.1,Protein shared split,ProtBERT SW,shared split,protein foundation embedding,SVM RBF,validation ROC-AUC,default_0.5,0.7274,0.7433,0.6667,0.3215,Best protein-only model on shared Phase 3 split
2,3,2.2,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,SVM RBF,diagnostic test comparison,default_0.5,0.6765,0.6504,0.6620,0.2933,"Diagnostic best genomic model, not official selection"
3,4,2.1,Genomic regulatory,K3 + K4 + Basic,handcrafted TSS-proximal 2kbUp+500bpDown,genomic handcrafted k-mer + GC/CpG features,Random Forest,validation ROC-AUC,default_0.5,0.6496,0.6327,0.6406,0.2557,Official genomic handcrafted model
4,5,1.2E,Protein,ProtBERT,sliding_window_1022_stride_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6487,0.6551,0.5896,0.1941,Best protein ranking representation before shared split
5,6,1.2D,Protein,ProtBERT,truncated_1022,protein foundation embedding,Logistic Regression,validation ROC-AUC,default_0.5,0.6371,0.6423,0.6014,0.1943,ProtBERT truncated
6,7,1.2B,Protein,ESM2_t6_8M,sliding_window_1022_stride_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.6277,0.6278,0.5870,0.1650,ESM-2 small sliding-window
7,8,1.2A,Protein,ESM2_t6_8M,truncated_1022,protein foundation embedding,Stacking,validation ROC-AUC,default_0.5,0.6202,0.6188,0.5926,0.1941,ESM-2 small truncated baseline
8,9,1.2C,Protein,ESM2_t12_35M,truncated_1022,protein foundation embedding,Soft Voting,validation ROC-AUC,default_0.5,0.5875,0.5942,0.5654,0.0995,Larger ESM-2 did not improve under truncation
9,10,1.1,Protein,AAC + Physchem,handcrafted,handcrafted protein descriptors,Random Forest,validation ROC-AUC,default_0.5,0.5520,0.5550,0.5390,0.0480,Protein handcrafted baseline


,gene_list_name,n_input_genes,n_significant_terms,n_t2d_relevant_terms,best_p_value_scientific,top_term,top_t2d_relevant_term
0,top_true_positives,86,175,63,6.97e-11,respiratory chain complex I,oxidative phosphorylation
1,top_50_combined_predictions,50,102,41,1.85e-05,NADH dehydrogenase (ubiquinone) activity,Oxidative phosphorylation
2,high_confidence_false_negatives,49,3,0,4.80e-02,Anterior uveitis,NaN
3,both_wrong_genes,74,1,0,1.07e-02,regulation of toll-like receptor 3 signaling pathway,NaN
4,high_confidence_false_positives,40,0,0,,NaN,NaN
5,combined_rescued_genes,18,0,0,,NaN,NaN


,gene_list_name,source,native,name,p_value_scientific,intersection_size,term_size,precision,recall,overlap_genes,matched_biological_groups
0,top_50_combined_predictions,KEGG,KEGG:00190,Oxidative phosphorylation,1.29e-02,5,136,0.1471,0.0368,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy;signaling_regulation
1,top_50_combined_predictions,WP,WP:WP623,Oxidative phosphorylation,1.48e-02,4,60,0.1111,0.0667,NDUFB2; NDUFV1; NDUFA7; NDUFB1,mitochondria_energy;signaling_regulation
2,top_50_combined_predictions,GO:BP,GO:0006119,oxidative phosphorylation,3.08e-02,5,146,0.1064,0.0342,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy;signaling_regulation
3,top_50_combined_predictions,GO:BP,GO:0031016,pancreas development,4.90e-02,4,79,0.0851,0.0506,NEUROD1; INSR; ISL1; PDX1,development_beta_cell;diabetes_glucose_insulin
4,top_50_combined_predictions,GO:MF,GO:0008137,NADH dehydrogenase (ubiquinone) activity,1.85e-05,5,41,0.1042,0.1220,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
5,top_50_combined_predictions,GO:CC,GO:0030964,NADH dehydrogenase complex,2.14e-05,5,49,0.1000,0.1020,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
6,top_50_combined_predictions,GO:CC,GO:0045271,respiratory chain complex I,2.14e-05,5,49,0.1000,0.1020,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
7,top_50_combined_predictions,GO:BP,GO:0006120,"mitochondrial electron transport, NADH to ubiquinone",2.00e-04,5,53,0.1064,0.0943,NDUFB2; NDUFV1; NDUFB3; NDUFA7; NDUFB1,mitochondria_energy
8,top_50_combined_predictions,GO:MF,GO:0001228,"DNA-binding transcription activator activity, RNA polymerase II-specific",2.91e-04,9,431,0.1875,0.0209,ETV1; NEUROD1; HHEX; TCF12; BCL11B; MEIS1; NFAT5; ISL1; TFAP2B,signaling_regulation
9,top_50_combined_predictions,GO:MF,GO:0001216,DNA-binding transcription activator activity,3.26e-04,9,437,0.1875,0.0206,ETV1; NEUROD1; HHEX; TCF12; BCL11B; MEIS1; NFAT5; ISL1; TFAP2B,signaling_regulation


,query_gene_list,curated_set,n_query_genes,n_curated_genes,n_overlap,overlap_fraction_query,overlap_genes
0,top_true_positives,known_T2D_GWAS_or_monogenic_related,86,28,8,0.0930,ABCC8; HHEX; IGF2BP2; INSR; KCNJ11; MTNR1B; NEUROD1; PDX1
1,top_true_positives,insulin_secretion_beta_cell,86,19,8,0.0930,ABCC8; CACNA1D; DPP4; GAD1; INSR; KCNJ11; NEUROD1; PDX1
2,top_50_combined_predictions,insulin_secretion_beta_cell,50,19,6,0.1200,CACNA1D; DPP4; GAD1; INSR; NEUROD1; PDX1
3,top_true_positives,mitochondrial_oxphos,86,17,6,0.0698,NDUFA2; NDUFA7; NDUFB1; NDUFB2; NDUFB3; NDUFV1
4,top_50_combined_predictions,known_T2D_GWAS_or_monogenic_related,50,28,5,0.1000,HHEX; IGF2BP2; INSR; NEUROD1; PDX1
5,top_50_combined_predictions,mitochondrial_oxphos,50,17,5,0.1000,NDUFA7; NDUFB1; NDUFB2; NDUFB3; NDUFV1
6,combined_rescued_genes,known_T2D_GWAS_or_monogenic_related,18,28,2,0.1111,ABCC8; MTNR1B
7,high_confidence_false_negatives,known_T2D_GWAS_or_monogenic_related,49,28,2,0.0408,PAX4; THADA
8,both_wrong_genes,known_T2D_GWAS_or_monogenic_related,74,28,2,0.0270,PAX4; THADA
9,combined_rescued_genes,insulin_secretion_beta_cell,18,19,1,0.0556,ABCC8


,n_top50_genes,n_true_positive,n_false_positive,n_with_curated_t2d_support,pct_with_curated_t2d_support,supported_genes
0,50,43,7,15,30.0,NDUFB2; NEUROD1; INSR; GAD1; NDUFV1; CACNA1D; HHEX; NDUFB3; SLC5A1; IGF2BP2; NDUFA7; SCD5; NDUFB1; DPP4; PDX1


,section,key_result,value,interpretation
0,Model performance,Official model,Official Combined Protein+Genomic SVM RBF,Selected multimodal model for final evaluation
1,Model performance,ROC-AUC / PR-AUC / F1 / MCC,0.7290 / 0.7573 / 0.6590 / 0.3438,Overall held-out test performance
2,Biological validation,Top true positives enrichment,175 significant terms; 63 T2D-relevant terms,Correct model predictions are biologically enriched
3,Biological validation,Top 50 predictions enrichment,102 significant terms; 41 T2D-relevant terms,Top-ranked predictions show pathway-level biological signal
4,Curated support,Top 50 curated T2D support,15/50 genes (30.0%),Top-ranked genes overlap with curated T2D/metabolic gene sets
5,Multimodal integration,Biologically supported rescued genes,ABCC8; MTNR1B,Some genes corrected by multimodal integration have known diabetes relevance
